In [9]:
import numpy as np
import pandas as pd
import jax.numpy as jnp
import jax 
import gc
from model_jax import Params, utility
from mh_jax import mh_sample

from tqdm import tqdm


In [10]:

q=pd.read_parquet('C:\BASIC_LEARNING\mls\code\data\q_d\q_d_d_f.parquet')
p=pd.read_parquet("C:\BASIC_LEARNING\mls\code\data\p_d\p_d_d_f.parquet")
L=q.shape[1]-2
p_l=p.iloc[-1,-L:]
week=p_l.name
q_l=q[q['WEEK_NO']==week].iloc[:,-L:]#household+week_no+L(14,)
q_np=q_l.mean(axis=0).values
p_np=p_l.values#(L,)
depart=q.iloc[:,2:].columns
del p,q 
gc.collect()


11

In [11]:
ws=[0,0.25,0.5,0.75,1]
ks=[2,5,10]
results=[]
for k in ks:
    for w in ws:
        samples=np.load(f"C:\BASIC_LEARNING\mls\code\JAX\samples\samples_K{k}_w{w}.npy")
        samples_mean=samples[-50000:,:].mean(axis=0)
        diff=(samples_mean-q_np)*p_np#money diff
        abs_diff=abs(diff).mean()
        square_diff=np.sqrt((diff**2).mean())
        results.append({
        "k": k,
        "w": w,
        "abs_diff": abs_diff,
        "square_diff": square_diff
        })
df_results=pd.DataFrame(results)


In [12]:
abs_diff=df_results.pivot_table(
    index='w',
    columns='k',
    values='abs_diff'
).reset_index()
abs_diff.to_csv('C:\\BASIC_LEARNING\\mls\\code\\JAX\\compare\\abs_diff.csv')
square_diff=df_results.pivot_table(
    index='w',
    columns='k',
    values='square_diff'
).reset_index()
square_diff.to_csv('C:\\BASIC_LEARNING\\mls\\code\\JAX\\compare\\square_diff.csv')


In [13]:
corr_results=[]
real_corr=(q_l.corr()).values
for k in ks:
    for w in ws:
        samples=np.load(f"C:\BASIC_LEARNING\mls\code\JAX\samples\samples_K{k}_w{w}.npy")
        subset = samples[-100_000::1000,:]
        corr=pd.DataFrame(subset).corr().values
        corr_diff=real_corr-corr
        abs_corr_diff=abs(corr_diff).mean()
        square_corr_diff=np.sqrt((corr_diff**2).mean())
        corr_results.append({
            "k":k,
            "w":w,
            'abs_corr_diff':abs_corr_diff,
            'square_corr_diff':square_corr_diff
        })
df_corr_results=pd.DataFrame(corr_results)


In [14]:
abs_corr_diff=df_corr_results.pivot_table(
    index='w',
    columns='k',
    values='abs_corr_diff'
).reset_index()
abs_corr_diff.to_csv('C:\\BASIC_LEARNING\\mls\\code\\JAX\\compare\\abs_corr_diff.csv')

square_corr_diff=df_corr_results.pivot_table(
    index='w',
    columns='k',
    values='square_corr_diff'
).reset_index()
square_corr_diff.to_csv('C:\\BASIC_LEARNING\\mls\\code\\JAX\\compare\\square_corr_diff.csv')


In [ ]:
abs_diff


k,w,2,5,10
0,0.00,2.615442,2.432617,2.098057
1,0.25,5.213019,3.198121,3.384855
2,0.50,21.139456,3.251102,3.394439
3,0.75,46.837494,1.599250,2.601991
4,1.00,6.476978,3.933512,3.896935


In [ ]:
square_diff


k,w,2,5,10
0,0.00,5.468679,4.736173,3.778729
1,0.25,10.382498,7.237567,8.012865
2,0.50,40.276274,6.684519,7.259486
3,0.75,84.305619,2.492715,4.343572
4,1.00,8.839843,8.824682,8.732782


In [ ]:
abs_corr_diff


k,w,2,5,10
0,0.00,0.134915,0.155805,0.135772
1,0.25,0.119158,0.120784,0.116317
2,0.50,0.143582,0.111252,0.123143
3,0.75,0.148725,0.118152,0.113802
4,1.00,0.128847,0.106701,0.116627


In [ ]:
square_corr_diff


k,w,2,5,10
0,0.00,0.176034,0.190231,0.173979
1,0.25,0.157013,0.161891,0.153581
2,0.50,0.192393,0.144532,0.152355
3,0.75,0.207667,0.150767,0.140197
4,1.00,0.175290,0.147249,0.157782
